In [1]:
# Force-reload all project modules from disk (avoids stale cached versions after edits).
import importlib, sys
for name in list(sys.modules):
    if name.startswith("src."):
        importlib.reload(sys.modules[name])

In [2]:
# Step 2 - Connect Colab to Google Drive.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Check that Colab can see the anukram folder and its files.
import os
os.listdir('/content/drive/MyDrive/anukram')

['src',
 'data',
 'notebooks',
 'requirements.txt',
 'app.py',
 'README.md',
 '.env',
 'artifacts']

In [4]:
# Step 3 - Set the project folder as the working directory.
import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/anukram"
os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)
print("Now working inside:", os.getcwd())

Now working inside: /content/drive/MyDrive/anukram


In [5]:
# Load the Groq key from the .env inside the project folder.
from dotenv import load_dotenv
import os
load_dotenv("/content/drive/MyDrive/anukram/.env")
key = os.getenv("GROQ_API_KEY")
print("Key loaded:", bool(key), "| starts with gsk_:", str(key).startswith("gsk_"))

Key loaded: True | starts with gsk_: True


In [6]:
# Step 5 - Install system OCR tools and all Python dependencies.
!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin poppler-utils
!pip install -q -r /content/drive/MyDrive/anukram/requirements.txt
print("Dependencies installed.")

Dependencies installed.


In [7]:
# Step 6 - Generate corpus, ingest documents, and run CONNECT.
from src.synthetic_corpus_generator import generate_corpus
from src.document_ingestion import load_and_chunk_documents, build_case_entity_index
from src.entity_resolution import resolve_entities
from src.entity_graph import build_entity_graph

generate_corpus()
chunks = load_and_chunk_documents()
index = build_case_entity_index(chunks)
resolution = resolve_entities(index)
graph = build_entity_graph(index, resolution)

for entity_id, rec in graph.items():
    tag = f"  [{rec['flag']}]" if rec["repeat_offender"] else ""
    print(f"{entity_id}: {rec['case_count']} case(s) across {rec['districts']}{tag}")
    for c in rec["timeline"]:
        print(f"     {c['year']}  {c['case_id']}  ({c['district']})  names_seen={c['names_seen']}")

2026-09-05 17:40:18 - [INFO] - anukram - Wrote synthetic case FIR/2013/DEL-A/00123 (ENT-00291)
INFO:anukram:Wrote synthetic case FIR/2013/DEL-A/00123 (ENT-00291)
2026-09-05 17:40:18 - [INFO] - anukram - Wrote synthetic case FIR/2017/GGN-B/00456 (ENT-00291)
INFO:anukram:Wrote synthetic case FIR/2017/GGN-B/00456 (ENT-00291)
2026-09-05 17:40:18 - [INFO] - anukram - Wrote synthetic case FIR/2019/FBD-C/00789 (ENT-00291)
INFO:anukram:Wrote synthetic case FIR/2019/FBD-C/00789 (ENT-00291)
2026-09-05 17:40:18 - [INFO] - anukram - Wrote synthetic case FIR/2020/DEL-D/00234 (ENT-00291)
INFO:anukram:Wrote synthetic case FIR/2020/DEL-D/00234 (ENT-00291)
2026-09-05 17:40:18 - [INFO] - anukram - Wrote synthetic case FIR/2018/DEL-E/00567 (ENT-00312)
INFO:anukram:Wrote synthetic case FIR/2018/DEL-E/00567 (ENT-00312)
2026-09-05 17:40:18 - [INFO] - anukram - Wrote synthetic case FIR/2019/DEL-F/00981 (ENT-00318)
INFO:anukram:Wrote synthetic case FIR/2019/DEL-F/00981 (ENT-00318)
2026-09-05 17:40:18 - [INFO]

RES-00003: 4 case(s) across ['Faridabad', 'Gurugram', 'New Delhi', 'South Delhi']  [REPEAT OFFENDER - BNS 71]
     2013  FIR/2013/DEL-A/00123  (South Delhi)  names_seen=['Suresh Kumar']
     2017  FIR/2017/GGN-B/00456  (Gurugram)  names_seen=['S. Kumar']
     2019  FIR/2019/FBD-C/00789  (Faridabad)  names_seen=['सुरेश कुमार']
     2020  FIR/2020/DEL-D/00234  (New Delhi)  names_seen=['Suresh Kmar']
RES-00002: 1 case(s) across ['East Delhi']
     2019  FIR/2019/DEL-F/00981  (East Delhi)  names_seen=['Ramesh Kumar']
RES-00001: 1 case(s) across ['West Delhi']
     2018  FIR/2018/DEL-E/00567  (West Delhi)  names_seen=['Suresh Sharma']


In [8]:
# Batch 2 - Build the bail-risk brief for the 2020 case.
import os, json
from src.config import ARTIFACTS_DIR
from src.entity_resolution import resolve_entities
from src.entity_graph import build_entity_graph
from src.bail_risk_brief import build_bail_brief, format_brief

with open(os.path.join(ARTIFACTS_DIR, "case_entities.json"), encoding="utf-8") as f:
    index = json.load(f)

graph = build_entity_graph(index, resolve_entities(index))

target_case = "FIR/2020/DEL-D/00234"
entity_id = next(e for e, r in graph.items()
                 if any(c["case_id"] == target_case for c in r["timeline"]))

brief = build_bail_brief(graph, index, entity_id, target_case, today="2020-07-20")
print(format_brief(brief))

2026-09-05 17:40:19 - [INFO] - anukram - Entity resolution: 6 cases -> 3 distinct persons.
INFO:anukram:Entity resolution: 6 cases -> 3 distinct persons.
2026-09-05 17:40:19 - [INFO] - anukram - Repeat-offender flag: RES-00003 across 4 cases in 4 districts.
INFO:anukram:Repeat-offender flag: RES-00003 across 4 cases in 4 districts.
2026-09-05 17:40:19 - [INFO] - anukram - Entity graph saved: 3 persons -> /content/drive/MyDrive/anukram/artifacts/entity_graph.json
INFO:anukram:Entity graph saved: 3 persons -> /content/drive/MyDrive/anukram/artifacts/entity_graph.json
2026-09-05 17:40:19 - [INFO] - anukram - Bail brief for RES-00003: risk=HIGH, priors=3.
INFO:anukram:Bail brief for RES-00003: risk=HIGH, priors=3.


BAIL-RISK BRIEF  (as of 2020-07-20)
Accused (resolved): RES-00003   Current case: FIR/2020/DEL-D/00234
Known name spellings: S. Kumar, Suresh Kmar, Suresh Kumar, सुरेश कुमार
------------------------------------------------------------
Prior linked cases: 3  across districts: Faridabad, Gurugram, New Delhi, South Delhi
    2013  FIR/2013/DEL-A/00123  (South Delhi)
    2017  FIR/2017/GGN-B/00456  (Gurugram)
    2019  FIR/2019/FBD-C/00789  (Faridabad)
REPEAT OFFENDER - BNS 71 applies
------------------------------------------------------------
Chargesheet deadline: 2020-09-08  (50 days left)  [GREEN]  (BNSS 193/187)
------------------------------------------------------------
RISK LEVEL: HIGH
Recommendation: Oppose bail. Accused shows a repeat pattern across multiple jurisdictions.
(Victim identities protected under BNS 72. Links are investigative leads, not proof of guilt.)
